# Notebook 05 — SQL Analysis

## 1. Import libraries and connect to MySQL

In [31]:
from pathlib import Path
from datetime import datetime, timezone
import json

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

# MySQL connection details
MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "root"
MYSQL_PASSWORD = "12345"

DATABASE_NAME = "global_seismic_trends"
TABLE_NAME = "earthquakes"
TABLE_SQL = f"`{TABLE_NAME}`"

database_url = URL.create(
    drivername="mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    database=DATABASE_NAME,
    query={"charset": "utf8mb4"},
)

engine = create_engine(
    database_url,
    pool_pre_ping=True,
    connect_args={"connect_timeout": 10},
)

# Validate the connection and retrieve table information
with engine.connect() as conn:
    mysql_version = conn.execute(
        text("SELECT VERSION()")
    ).scalar()

    current_database = conn.execute(
        text("SELECT DATABASE()")
    ).scalar()

    row_count = conn.execute(
        text(f"SELECT COUNT(*) FROM {TABLE_SQL}")
    ).scalar()

    columns = conn.execute(
        text(f"SHOW COLUMNS FROM {TABLE_SQL}")
    ).fetchall()

column_names = [row[0] for row in columns]

print("MySQL connection successful!")
print("MySQL version:", mysql_version)
print("Database:", current_database)
print(f"Rows in {TABLE_NAME}: {row_count:,}")
print("Columns:", len(column_names))

MySQL connection successful!
MySQL version: 8.0.45
Database: global_seismic_trends
Rows in earthquakes: 739,554
Columns: 51


## 2. Verify table schema

In [32]:
schema_df = pd.DataFrame(
    columns,
    columns=["column_name", "data_type", "nullable", "key", "default", "extra"]
)
display(schema_df)

required_columns = {
    "id", "time", "latitude", "longitude", "depth_km", "mag",
    "magType", "place", "status", "tsunami", "sig", "net",
    "nst", "dmin", "rms", "gap", "type", "types"
}
missing_required = sorted(required_columns - set(column_names))
if missing_required:
    raise ValueError(
        "The earthquakes table is missing required columns: "
        + ", ".join(missing_required)
        + ". Verify that Notebook 04 imported the expected cleaned dataset."
    )
else:
    print("All expected core analysis fields are present.")


,column_name,data_type,nullable,key,default,extra
0,id,text,YES,,None,
1,time,datetime,YES,,None,
2,updated,datetime,YES,,None,
3,latitude,double,YES,,None,
4,longitude,double,YES,,None,
5,depth_km,double,YES,,None,
6,mag,double,YES,,None,
7,magType,text,YES,,None,
8,place,text,YES,,None,
9,status,text,YES,,None,


All expected core analysis fields are present.


## 3. Helper function

In [33]:
def run_query(sql):
    """Run a read-only SQL query and return a pandas DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)


## 4. Task 1 — Top 10 strongest earthquakes


In [34]:
top_10_strongest = run_query(f'''
SELECT id, time, mag, magType, depth_km, place, latitude, longitude, tsunami, sig
FROM {TABLE_SQL}
WHERE mag IS NOT NULL
ORDER BY mag DESC, time DESC, id
LIMIT 10
''')
display(top_10_strongest)


,id,time,mag,magType,depth_km,place,latitude,longitude,tsunami,sig
0,us6000qw60,2025-07-29 23:24:52,8.8,mww,35.000,"2025 Kamchatka Peninsula, Russia Earthquake",52.4948,160.2395,1,2114
1,us6000tkt2,2026-08-14 21:58:22,7.8,mww,10.000,"66 km NNW of Ende, Indonesia",-8.3358,121.3446,0,1059
2,us7000srb1,2026-06-07 23:37:42,7.8,mww,56.991,"25 km SW of Kablalan, Philippines",5.5994,125.0560,0,1571
3,us7000qx2g,2025-09-18 18:58:15,7.8,mww,27.000,"140 km E of Petropavlovsk-Kamchatsky, Russia",53.1426,160.7206,1,1021
4,us6000jllz,2023-02-06 01:17:34,7.8,mww,10.000,"Pazarcik earthquake, Kahramanmaras earthquake sequence",37.2256,37.0143,0,2910
5,us6000u18k,2026-10-09 17:56:06,7.7,mww,12.647,"12 km WSW of Pitaloza Arriba, Panama",7.5868,-80.7690,0,1790
6,us7000pn9s,2025-03-28 06:20:53,7.7,mww,10.000,"2025 Mandalay, Burma (Myanmar) Earthquake",22.0110,95.9363,0,2910
7,us6000kd0n,2023-05-19 02:57:03,7.7,mww,18.053,southeast of the Loyalty Islands,-23.2063,170.7423,1,935
8,us6000rtdt,2025-12-08 14:15:10,7.6,mww,40.720,"2025 Aomori Prefecture, Japan Earthquake",40.9984,142.1836,1,1122
9,us6000rgf4,2025-10-10 20:29:20,7.6,mww,5.639,Drake Passage,-60.1875,-61.8164,1,893


## 5. Task 2 — Top 10 deepest earthquakes

In [35]:
top_10_deepest = run_query(f'''
SELECT id, time, depth_km, mag, magType, place, latitude, longitude, net, nst
FROM {TABLE_SQL}
WHERE depth_km IS NOT NULL
  AND depth_km >= 0
ORDER BY depth_km DESC, time DESC, id
LIMIT 10
''')
display(top_10_deepest)


,id,time,depth_km,mag,magType,place,latitude,longitude,net,nst
0,us6000sp3p,2026-04-02 16:59:18,683.578,4.0,mb,"206 km ENE of Sola, Vanuatu",-12.9316,169.1934,us,26.0
1,us6000k2db,2023-04-01 18:09:17,681.238,4.0,mb,"208 km ENE of Sola, Vanuatu",-13.1462,169.3241,us,53.0
2,us7000kxdn,2023-09-18 15:35:27,675.265,4.2,mb,Vanuatu region,-15.1098,171.6867,us,24.0
3,us6000mivr,2024-03-08 22:42:24,671.043,4.2,mb,Fiji region,-19.3114,-177.8158,us,16.0
4,us6000rk66,2025-10-29 17:07:34,669.556,4.8,mb,"205 km ESE of Levuka, Fiji",-19.0745,-179.0467,us,54.0
5,us7000q1jk,2025-05-10 02:03:17,667.237,4.2,mb,"299 km E of Levuka, Fiji",-17.8425,-177.8703,us,19.0
6,us7000s0st,2026-02-14 08:27:48,667.197,4.3,mb,"205 km ESE of Levuka, Fiji",-19.0361,-179.0278,us,33.0
7,us6000ta9m,2026-07-05 14:22:25,665.993,5.7,mwb,"274 km SE of Levuka, Fiji",-20.0123,-179.0709,us,85.0
8,us6000tknp,2026-08-10 16:50:10,665.326,4.0,mb,"178 km NE of Sola, Vanuatu",-12.9179,168.8726,us,43.0
9,us6000txkp,2026-09-20 02:50:10,665.057,4.2,mb,Vanuatu region,-14.5749,170.4038,us,20.0


## 6. Task 3 — Shallow earthquakes below 50 km with magnitude above 7.5

In [36]:
shallow_high_magnitude = run_query(f'''
SELECT id, time, mag, depth_km, magType, place, latitude, longitude, tsunami, sig
FROM {TABLE_SQL}
WHERE depth_km IS NOT NULL
  AND depth_km < 50
  AND mag IS NOT NULL
  AND mag > 7.5
ORDER BY mag DESC, time DESC
''')
display(shallow_high_magnitude)
print("Matching events:", len(shallow_high_magnitude))


,id,time,mag,depth_km,magType,place,latitude,longitude,tsunami,sig
0,us6000qw60,2025-07-29 23:24:52,8.8,35.000,mww,"2025 Kamchatka Peninsula, Russia Earthquake",52.4948,160.2395,1,2114
1,us6000tkt2,2026-08-14 21:58:22,7.8,10.000,mww,"66 km NNW of Ende, Indonesia",-8.3358,121.3446,0,1059
2,us7000qx2g,2025-09-18 18:58:15,7.8,27.000,mww,"140 km E of Petropavlovsk-Kamchatsky, Russia",53.1426,160.7206,1,1021
3,us6000jllz,2023-02-06 01:17:34,7.8,10.000,mww,"Pazarcik earthquake, Kahramanmaras earthquake sequence",37.2256,37.0143,0,2910
4,us6000u18k,2026-10-09 17:56:06,7.7,12.647,mww,"12 km WSW of Pitaloza Arriba, Panama",7.5868,-80.7690,0,1790
5,us7000pn9s,2025-03-28 06:20:53,7.7,10.000,mww,"2025 Mandalay, Burma (Myanmar) Earthquake",22.0110,95.9363,0,2910
6,us6000kd0n,2023-05-19 02:57:03,7.7,18.053,mww,southeast of the Loyalty Islands,-23.2063,170.7423,1,935
7,us6000rtdt,2025-12-08 14:15:10,7.6,40.720,mww,"2025 Aomori Prefecture, Japan Earthquake",40.9984,142.1836,1,1122
8,us6000rgf4,2025-10-10 20:29:20,7.6,5.639,mww,Drake Passage,-60.1875,-61.8164,1,893
9,us7000pcdl,2025-02-08 23:23:15,7.6,14.326,mww,"210 km SSW of George Town, Cayman Islands",17.6506,-82.3950,1,1182


Matching events: 12


## 7. Task 5 — Average magnitude by magnitude type (`magType`)


In [37]:
average_magnitude_by_type = run_query(f'''
SELECT
    COALESCE(NULLIF(TRIM(magType), ''), 'Missing') AS magnitude_type,
    COUNT(*) AS event_count,
    COUNT(mag) AS events_with_magnitude,
    ROUND(AVG(mag), 3) AS average_magnitude,
    MIN(mag) AS minimum_magnitude,
    MAX(mag) AS maximum_magnitude
FROM {TABLE_SQL}
GROUP BY COALESCE(NULLIF(TRIM(magType), ''), 'Missing')
ORDER BY average_magnitude DESC
''')
display(average_magnitude_by_type)


,magnitude_type,event_count,events_with_magnitude,average_magnitude,minimum_magnitude,maximum_magnitude
0,mwc,1,1,7.000,7.00,7.00000
1,Mi,1,1,6.600,6.60,6.60000
2,ms_20,1,1,5.800,5.80,5.80000
3,mwb,26,26,5.758,5.10,7.00000
4,mww,6504,6504,5.350,4.40,8.80000
5,mwp,1,1,5.250,5.25,5.25000
6,ms_vx,5,5,4.500,3.90,5.20000
7,mb,73276,73276,4.417,3.00,6.50000
8,mwr,2514,2514,4.315,2.90,6.00000
9,mw,548,548,3.920,3.01,7.00000


## 8. Task 6 — Year with the most earthquakes


In [38]:
yearly_counts = run_query(f'''
SELECT YEAR(time) AS event_year, COUNT(*) AS earthquake_count
FROM {TABLE_SQL}
WHERE time IS NOT NULL
GROUP BY YEAR(time)
ORDER BY event_year
''')
display(yearly_counts)

if not yearly_counts.empty:
    peak_year = yearly_counts.loc[yearly_counts["earthquake_count"].idxmax()]
    print(f"Highest count: {int(peak_year['event_year'])} "
          f"({int(peak_year['earthquake_count']):,} records; calendar-year coverage may be partial).")


,event_year,earthquake_count
0,2021,26767
1,2022,150752
2,2023,151618
3,2024,155899
4,2025,140351
5,2026,114167


Highest count: 2024 (155,899 records; calendar-year coverage may be partial).


## 9. Task 7 — Month with the highest number of earthquakes


In [ ]:

monthly_counts_by_name = run_query(f"""
SELECT
    MONTH(time) AS month_number,
    MONTHNAME(time) AS month_name,
    COUNT(*) AS earthquake_count
FROM {TABLE_SQL}
WHERE time IS NOT NULL
GROUP BY MONTH(time), MONTHNAME(time)
ORDER BY earthquake_count DESC
""")

print("Earthquake counts by calendar month:")
display(monthly_counts_by_name)


monthly_counts_by_period = run_query(f"""
SELECT
    DATE_FORMAT(time, '%Y-%m') AS `year_month`,
    COUNT(*) AS earthquake_count
FROM {TABLE_SQL}
WHERE time IS NOT NULL
GROUP BY DATE_FORMAT(time, '%Y-%m')
ORDER BY earthquake_count DESC, `year_month`
LIMIT 12
""")

print("Top 12 year-month periods:")
display(monthly_counts_by_period)

Earthquake counts by calendar month:


,month_number,month_name,earthquake_count
0,7,July,67172
1,3,March,66552
2,8,August,63895
3,12,December,63666
4,4,April,63104
5,1,January,62396
6,5,May,60627
7,2,February,60313
8,6,June,59915
9,9,September,59710


Top 12 year-month periods:


,year_month,earthquake_count
0,2025-07,16543
1,2023-03,15792
2,2024-02,14980
3,2024-04,14055
4,2025-12,13898
5,2024-03,13870
6,2024-05,13654
7,2023-04,13349
8,2024-07,13219
9,2024-06,13199


## 10. Task 8 — Day of week with the most earthquakes

In [40]:
weekday_counts = run_query(f'''
SELECT
    DAYOFWEEK(time) AS weekday_number,
    DAYNAME(time) AS weekday_name,
    COUNT(*) AS earthquake_count
FROM {TABLE_SQL}
WHERE time IS NOT NULL
GROUP BY DAYOFWEEK(time), DAYNAME(time)
ORDER BY earthquake_count DESC
''')
display(weekday_counts)


,weekday_number,weekday_name,earthquake_count
0,4,Wednesday,106762
1,5,Thursday,106702
2,2,Monday,105985
3,6,Friday,105916
4,3,Tuesday,105633
5,1,Sunday,104465
6,7,Saturday,104091


## 11. Task 9 — Number of earthquakes per hour (UTC)

In [41]:
hourly_counts = run_query(f'''
SELECT HOUR(time) AS hour_utc, COUNT(*) AS earthquake_count
FROM {TABLE_SQL}
WHERE time IS NOT NULL
GROUP BY HOUR(time)
ORDER BY hour_utc
''')
display(hourly_counts)


,hour_utc,earthquake_count
0,0,30555
1,1,30190
2,2,30837
3,3,30992
4,4,31383
5,5,31467
6,6,31684
7,7,32043
8,8,32403
9,9,32219


## 12. Task 10 — Most active reporting network (`net`)

In [42]:
network_activity = run_query(f'''
SELECT
    COALESCE(NULLIF(TRIM(net), ''), 'Missing') AS reporting_network,
    COUNT(*) AS earthquake_count,
    COUNT(mag) AS events_with_magnitude,
    ROUND(AVG(mag), 3) AS average_magnitude,
    MIN(time) AS first_event_utc,
    MAX(time) AS last_event_utc
FROM {TABLE_SQL}
GROUP BY COALESCE(NULLIF(TRIM(net), ''), 'Missing')
ORDER BY earthquake_count DESC
''')
display(network_activity.head(25))
if not network_activity.empty:
    print("Most active network:", network_activity.iloc[0]["reporting_network"],
          f"({int(network_activity.iloc[0]['earthquake_count']):,} records)")


,reporting_network,earthquake_count,events_with_magnitude,average_magnitude,first_event_utc,last_event_utc
0,ak,177202,177195,1.640,2021-10-22 13:44:12,2026-10-22 13:12:09
1,nc,114024,113984,1.013,2021-10-22 13:22:28,2026-10-22 13:03:02
2,us,109357,109352,4.038,2021-10-22 14:06:00,2026-10-22 12:22:58
3,ci,84093,84093,1.100,2021-10-22 13:55:21,2026-10-22 12:49:04
4,av,57612,57612,0.159,2021-10-22 20:30:47,2026-10-21 22:31:18
5,hv,37600,37600,1.908,2021-10-22 16:03:18,2026-10-22 13:13:42
6,tx,36877,36877,1.675,2021-10-22 22:33:46,2026-10-22 08:20:00
7,nn,28591,28591,0.986,2021-10-22 14:19:01,2026-10-22 00:58:26
8,ok,22631,22629,0.505,2021-10-22 15:54:21,2026-10-21 13:55:26
9,uw,20748,20748,0.903,2021-10-22 17:07:12,2026-10-21 21:04:07


Most active network: ak (177,202 records)


## 13. Task 11 — Reviewed versus automatic events (`status`)

In [43]:
status_counts = run_query(f'''
SELECT
    COALESCE(NULLIF(TRIM(status), ''), 'Missing') AS review_status,
    COUNT(*) AS earthquake_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM {TABLE_SQL}), 2) AS percent_of_records
FROM {TABLE_SQL}
GROUP BY COALESCE(NULLIF(TRIM(status), ''), 'Missing')
ORDER BY earthquake_count DESC
''')
display(status_counts)


,review_status,earthquake_count,percent_of_records
0,reviewed,657863,88.95
1,automatic,81691,11.05


## 14. Task 12 — Count of earthquakes by event type (`type`)


In [44]:
event_type_counts = run_query(f'''
SELECT
    COALESCE(NULLIF(TRIM(type), ''), 'Missing') AS event_type,
    COUNT(*) AS earthquake_count
FROM {TABLE_SQL}
GROUP BY COALESCE(NULLIF(TRIM(type), ''), 'Missing')
ORDER BY earthquake_count DESC
''')
display(event_type_counts)


,event_type,earthquake_count
0,earthquake,722294
1,quarry blast,7105
2,explosion,5067
3,ice quake,3699
4,mining explosion,1044
5,other event,263
6,Landslide,42
7,volcanic eruption,12
8,sonic boom,9
9,chemical explosion,6


## 15. Task 13 — Count earthquakes by data-type tokens (`types`)



In [45]:
data_type_token_counts = run_query(f'''
WITH RECURSIVE split_types AS (
    SELECT
        id,
        TRIM(SUBSTRING_INDEX(types, ',', 1)) AS type_token,
        CASE
            WHEN types LIKE '%,%' THEN SUBSTRING(types, LOCATE(',', types) + 1)
            ELSE NULL
        END AS remaining_types
    FROM {TABLE_SQL}
    WHERE types IS NOT NULL AND TRIM(types) <> ''

    UNION ALL

    SELECT
        id,
        TRIM(SUBSTRING_INDEX(remaining_types, ',', 1)) AS type_token,
        CASE
            WHEN remaining_types LIKE '%,%' THEN SUBSTRING(remaining_types, LOCATE(',', remaining_types) + 1)
            ELSE NULL
        END AS remaining_types
    FROM split_types
    WHERE remaining_types IS NOT NULL AND TRIM(remaining_types) <> ''
)
SELECT
    type_token,
    COUNT(DISTINCT id) AS earthquake_count
FROM split_types
WHERE type_token <> ''
GROUP BY type_token
ORDER BY earthquake_count DESC, type_token
''')
display(data_type_token_counts)


,type_token,earthquake_count
0,origin,739554
1,phase-data,739193
2,nearby-cities,198389
3,scitech-link,157894
4,dyfi,36855
5,focal-mechanism,32904
6,shakemap,9563
7,earthquake-name,8291
8,moment-tensor,8049
9,internal-moment-tensor,4240


## 16. Task 14 — Events with high station coverage (`nst`)


In [46]:
NST_THRESHOLD = 20

high_station_coverage_summary = run_query(f'''
SELECT
    COUNT(*) AS events_above_threshold,
    COUNT(mag) AS events_with_magnitude,
    ROUND(AVG(nst), 2) AS average_station_count,
    MIN(nst) AS minimum_station_count,
    MAX(nst) AS maximum_station_count
FROM {TABLE_SQL}
WHERE nst > {int(NST_THRESHOLD)}
''')
display(high_station_coverage_summary)

high_station_coverage_events = run_query(f'''
SELECT id, time, mag, nst, net, magType, place, latitude, longitude
FROM {TABLE_SQL}
WHERE nst > {int(NST_THRESHOLD)}
ORDER BY nst DESC, mag DESC
LIMIT 100
''')
display(high_station_coverage_events)


,events_above_threshold,events_with_magnitude,average_station_count,minimum_station_count,maximum_station_count
0,244350,244332,42.03,21.0,619.0


,id,time,mag,nst,net,magType,place,latitude,longitude
0,us6000m12f,2024-01-02 01:17:32,5.4,619.0,us,mww,"11 km W of Anamizu, Japan",37.2470,136.7689
1,us6000qzfl,2025-08-09 08:01:41,5.0,566.0,us,mww,"120 km ENE of Ozernovskiy, Russia",51.7700,158.1810
2,us7000rluk,2026-01-01 06:46:55,5.7,516.0,us,mww,"111 km N of Yakutat, Alaska",60.5362,-140.0494
3,us7000pvtr,2025-04-29 14:53:38,6.8,475.0,us,mww,Macquarie Island region,-54.2616,155.6446
4,usd001097k,2023-12-11 18:36:00,5.5,466.0,us,mww,"49 km WNW of San Antonio de los Cobres, Argentina",-24.0849,-66.7812
...,...,...,...,...,...,...,...,...,...
95,us7000qd00,2025-07-16 14:21:52,5.0,306.0,us,mww,"57 km E of Atka, Alaska",52.1732,-173.3679
96,us6000t04s,2026-05-25 21:52:20,6.9,305.0,us,mww,"34 km ENE of Calama, Chile",-22.3667,-68.6013
97,us6000mzme,2024-05-19 09:35:25,6.0,305.0,us,mww,"169 km SW of Nikolski, Alaska",52.0681,-170.9122
98,us6000j8hb,2022-12-10 22:11:06,5.2,305.0,us,mww,"177 km SE of Attu Station, Alaska",51.9179,175.3114


## 17. Task 15 — Tsunami-flagged events per year

In [47]:
tsunami_by_year = run_query(f'''
SELECT
    YEAR(time) AS event_year,
    COUNT(*) AS total_earthquakes,
    SUM(CASE WHEN tsunami = 1 THEN 1 ELSE 0 END) AS tsunami_flagged_events
FROM {TABLE_SQL}
WHERE time IS NOT NULL
GROUP BY YEAR(time)
ORDER BY event_year
''')
display(tsunami_by_year)


,event_year,total_earthquakes,tsunami_flagged_events
0,2021,26767,25.0
1,2022,150752,136.0
2,2023,151618,119.0
3,2024,155899,114.0
4,2025,140351,142.0
5,2026,114167,49.0


## 18. Task 16 — Count earthquakes by alert level




In [48]:
if "alert" in column_names:
    alert_level_counts = run_query(f'''
    SELECT
        COALESCE(NULLIF(TRIM(alert), ''), 'Missing / not assigned') AS alert_level,
        COUNT(*) AS earthquake_count
    FROM {TABLE_SQL}
    GROUP BY COALESCE(NULLIF(TRIM(alert), ''), 'Missing / not assigned')
    ORDER BY earthquake_count DESC
    ''')
    display(alert_level_counts)
else:
    alert_level_counts = pd.DataFrame()
    print("Skipped: the MySQL table has no 'alert' column. The task requires a source field that provides alert levels.")


,alert_level,earthquake_count
0,Missing / not assigned,735465
1,green,3918
2,yellow,123
3,orange,25
4,red,23


## 19. Task 17 — Year-over-year growth in global earthquake count


In [49]:
year_over_year_growth = run_query(f'''
WITH yearly AS (
    SELECT YEAR(time) AS event_year, COUNT(*) AS earthquake_count
    FROM {TABLE_SQL}
    WHERE time IS NOT NULL
    GROUP BY YEAR(time)
),
with_previous AS (
    SELECT
        event_year,
        earthquake_count,
        LAG(earthquake_count) OVER (ORDER BY event_year) AS previous_year_count
    FROM yearly
)
SELECT
    event_year,
    earthquake_count,
    previous_year_count,
    ROUND(
        (earthquake_count - previous_year_count) * 100.0 / NULLIF(previous_year_count, 0),
        2
    ) AS yoy_growth_percent
FROM with_previous
ORDER BY event_year
''')
display(year_over_year_growth)


,event_year,earthquake_count,previous_year_count,yoy_growth_percent
0,2021,26767,NaN,NaN
1,2022,150752,26767.0,463.20
2,2023,151618,150752.0,0.57
3,2024,155899,151618.0,2.82
4,2025,140351,155899.0,-9.97
5,2026,114167,140351.0,-18.66


## 20. Task 18 — Average magnitude difference for tsunami-flagged versus non-flagged events


In [50]:
magnitude_by_tsunami_flag = run_query(f'''
SELECT
    CASE
        WHEN tsunami = 1 THEN 'Tsunami flag = 1'
        WHEN tsunami = 0 THEN 'Tsunami flag = 0'
        ELSE 'Missing / other'
    END AS tsunami_group,
    COUNT(*) AS event_count,
    COUNT(mag) AS events_with_magnitude,
    ROUND(AVG(mag), 3) AS average_magnitude
FROM {TABLE_SQL}
GROUP BY tsunami_group
ORDER BY tsunami_group
''')
display(magnitude_by_tsunami_flag)

valid_groups = magnitude_by_tsunami_flag[
    magnitude_by_tsunami_flag["tsunami_group"].isin(["Tsunami flag = 1", "Tsunami flag = 0"])
].set_index("tsunami_group")
if {"Tsunami flag = 1", "Tsunami flag = 0"}.issubset(set(valid_groups.index)):
    flagged_avg = valid_groups.loc["Tsunami flag = 1", "average_magnitude"]
    unflagged_avg = valid_groups.loc["Tsunami flag = 0", "average_magnitude"]
    if pd.notna(flagged_avg) and pd.notna(unflagged_avg):
        print(f"Average magnitude difference (flagged minus unflagged): {flagged_avg - unflagged_avg:.3f}")


,tsunami_group,event_count,events_with_magnitude,average_magnitude
0,Tsunami flag = 0,738969,738912,1.655
1,Tsunami flag = 1,585,585,5.383


Average magnitude difference (flagged minus unflagged): 3.728


## 21. Task 19 — Data reliability proxy using `gap` and `rms`


In [51]:
MIN_NETWORK_RECORDS = 100

network_quality = run_query(f'''
SELECT
    COALESCE(NULLIF(TRIM(net), ''), 'Missing') AS reporting_network,
    COUNT(*) AS event_count,
    COUNT(gap) AS events_with_gap,
    ROUND(AVG(gap), 3) AS average_gap,
    COUNT(rms) AS events_with_rms,
    ROUND(AVG(rms), 4) AS average_rms,
    ROUND(AVG(CASE WHEN gap IS NOT NULL AND rms IS NOT NULL THEN gap END), 3) AS mean_gap_when_present
FROM {TABLE_SQL}
GROUP BY COALESCE(NULLIF(TRIM(net), ''), 'Missing')
HAVING COUNT(*) >= {int(MIN_NETWORK_RECORDS)}
ORDER BY average_gap DESC, average_rms DESC
''')
display(network_quality)


,reporting_network,event_count,events_with_gap,average_gap,events_with_rms,average_rms,mean_gap_when_present
0,pr,17740,17740,211.434,17740,0.1876,211.434
1,av,57612,57611,176.205,57611,0.1424,176.205
2,nn,28591,28591,151.128,28587,0.1315,151.136
3,uw,20748,20745,130.143,20745,0.1608,130.143
4,us,109357,109341,127.580,109357,0.6668,127.580
5,hv,37600,37598,127.312,37598,0.1391,127.312
6,uu,20206,20206,122.167,20206,0.1251,122.167
7,mb,10220,10220,120.661,10220,0.1338,120.661
8,ak,177202,29224,113.185,177202,0.5648,113.185
9,ok,22631,22631,108.632,22631,0.1950,108.632


## 22. Task 20 — Consecutive earthquake records within 50 km and within one hour


In [52]:
# Task 29 — Consecutive Nearby Earthquakes
# Find consecutive events occurring within 50 km and 1 hour.

consecutive_nearby_events = run_query(f"""
WITH ordered_events AS (
    SELECT
        id,
        time,
        latitude,
        longitude,
        LAG(id) OVER (ORDER BY time, id) AS previous_id,
        LAG(time) OVER (ORDER BY time, id) AS previous_time,
        LAG(latitude) OVER (ORDER BY time, id) AS previous_latitude,
        LAG(longitude) OVER (ORDER BY time, id) AS previous_longitude
    FROM {TABLE_SQL}
    WHERE time IS NOT NULL
      AND latitude BETWEEN -90 AND 90
      AND longitude BETWEEN -180 AND 180
),
distances AS (
    SELECT
        id,
        time,
        latitude,
        longitude,
        previous_id,
        previous_time,
        previous_latitude,
        previous_longitude,
        TIMESTAMPDIFF(
            SECOND, previous_time, time
        ) AS seconds_apart,
        6371 * ACOS(
            LEAST(1.0, GREATEST(-1.0,
                SIN(RADIANS(latitude))
                * SIN(RADIANS(previous_latitude))
                + COS(RADIANS(latitude))
                * COS(RADIANS(previous_latitude))
                * COS(RADIANS(longitude - previous_longitude))
            ))
        ) AS distance_km
    FROM ordered_events
    WHERE previous_time IS NOT NULL
      AND previous_latitude IS NOT NULL
      AND previous_longitude IS NOT NULL
)
SELECT
    previous_id,
    previous_time,
    id AS current_id,
    time AS event_time,
    seconds_apart,
    ROUND(distance_km, 3) AS distance_km,
    previous_latitude,
    previous_longitude,
    latitude AS current_latitude,
    longitude AS current_longitude
FROM distances
WHERE seconds_apart BETWEEN 0 AND 3600
  AND distance_km <= 50
ORDER BY event_time
""")

print("Consecutive nearby earthquake events:")
display(consecutive_nearby_events)

print(
    f"Total matching consecutive event pairs: "
    f"{len(consecutive_nearby_events):,}"
)

Consecutive nearby earthquake events:


,previous_id,previous_time,current_id,event_time,seconds_apart,distance_km,previous_latitude,previous_longitude,current_latitude,current_longitude
0,mb80527769,2021-10-22 13:27:15,mb80527774,2021-10-22 13:31:59,284,40.784,47.839167,-114.289000,47.766833,-113.753667
1,nc73642750,2021-10-22 13:59:38,nc73642760,2021-10-22 14:02:34,176,0.024,38.512500,-119.402333,38.512333,-119.402500
2,nc71126574,2021-10-22 14:09:48,nc73642765,2021-10-22 14:10:26,38,0.734,38.520500,-119.407833,38.516000,-119.401667
3,nc73642765,2021-10-22 14:10:26,nc73642775,2021-10-22 14:12:45,139,1.748,38.516000,-119.401667,38.531667,-119.403333
4,nc73642780,2021-10-22 14:29:48,nc73642785,2021-10-22 14:34:01,253,1.740,38.827331,-122.812836,38.816502,-122.798332
...,...,...,...,...,...,...,...,...,...,...
58445,nc75453316,2026-10-22 06:09:13,nc75453326,2026-10-22 06:22:01,768,4.712,38.821335,-122.812668,38.797165,-122.767998
58446,nc75453326,2026-10-22 06:22:01,nc75453331,2026-10-22 06:28:39,398,3.393,38.797165,-122.767998,38.818501,-122.795998
58447,nc75453351,2026-10-22 10:16:33,nc75453356,2026-10-22 10:17:04,31,2.836,38.828499,-122.807999,38.803001,-122.808670
58448,aka2026uxypcr,2026-10-22 12:04:19,aka2026uxyuue,2026-10-22 12:10:46,387,36.373,58.240000,-155.059000,57.913000,-155.075000


Total matching consecutive event pairs: 58,450


## 23. Task 21 — Regions with the highest frequency of deep-focus earthquakes (`depth_km > 300`)


In [53]:
deep_focus_coordinate_bins = run_query(f'''
SELECT
    FLOOR(latitude * 2) / 2 AS latitude_bin_start,
    FLOOR(longitude * 2) / 2 AS longitude_bin_start,
    COUNT(*) AS deep_focus_event_count,
    ROUND(AVG(mag), 3) AS average_magnitude,
    MAX(mag) AS maximum_magnitude,
    ROUND(AVG(depth_km), 2) AS average_depth_km
FROM {TABLE_SQL}
WHERE depth_km > 300
  AND latitude BETWEEN -90 AND 90
  AND longitude BETWEEN -180 AND 180
GROUP BY FLOOR(latitude * 2) / 2, FLOOR(longitude * 2) / 2
ORDER BY deep_focus_event_count DESC
LIMIT 30
''')
display(deep_focus_coordinate_bins)


,latitude_bin_start,longitude_bin_start,deep_focus_event_count,average_magnitude,maximum_magnitude,average_depth_km
0,-18.0,-179.0,318,4.337,5.9,561.63
1,-24.0,-180.0,220,4.407,5.7,521.37
2,-18.0,-178.5,217,4.324,5.7,572.27
3,-20.5,-178.0,199,4.304,5.5,509.25
4,-18.5,-178.5,192,4.327,5.6,569.35
5,-18.5,-178.0,147,4.301,5.9,579.61
6,-23.5,-180.0,137,4.414,5.6,540.09
7,-24.5,-180.0,119,4.359,5.4,506.03
8,-21.0,-179.0,118,4.413,5.9,585.93
9,-19.0,-178.0,117,4.305,5.6,560.16


## 24. Additional useful summaries retained from the original notebook

These tables support validation and dashboard planning.


In [54]:
overall_kpis = run_query(f'''
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT id) AS unique_event_ids,
    COUNT(mag) AS records_with_magnitude,
    SUM(CASE WHEN mag >= 5 THEN 1 ELSE 0 END) AS magnitude_5_plus,
    SUM(CASE WHEN mag >= 7 THEN 1 ELSE 0 END) AS magnitude_7_plus,
    SUM(CASE WHEN tsunami = 1 THEN 1 ELSE 0 END) AS tsunami_flagged_events,
    MIN(time) AS earliest_event_utc,
    MAX(time) AS latest_event_utc
FROM {TABLE_SQL}
''')
display(overall_kpis)

magnitude_bands = run_query(f'''
SELECT
    CASE
        WHEN mag IS NULL THEN 'Missing'
        WHEN mag < 2 THEN 'Below 2.0'
        WHEN mag < 3 THEN '2.0-2.9'
        WHEN mag < 4 THEN '3.0-3.9'
        WHEN mag < 5 THEN '4.0-4.9'
        WHEN mag < 6 THEN '5.0-5.9'
        WHEN mag < 7 THEN '6.0-6.9'
        ELSE '7.0+'
    END AS magnitude_band,
    COUNT(*) AS event_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM {TABLE_SQL}), 2) AS percent_of_records
FROM {TABLE_SQL}
GROUP BY
    CASE
        WHEN mag IS NULL THEN 'Missing'
        WHEN mag < 2 THEN 'Below 2.0'
        WHEN mag < 3 THEN '2.0-2.9'
        WHEN mag < 4 THEN '3.0-3.9'
        WHEN mag < 5 THEN '4.0-4.9'
        WHEN mag < 6 THEN '5.0-5.9'
        WHEN mag < 7 THEN '6.0-6.9'
        ELSE '7.0+'
    END
ORDER BY MIN(mag)
''')
display(magnitude_bands)

# Depth categories — count events and calculate average magnitude

depth_categories = run_query(f"""
SELECT
    CASE
        WHEN depth_km IS NULL THEN 'Missing'
        WHEN depth_km < 0 THEN 'Review - negative'
        WHEN depth_km < 70 THEN 'Shallow (<70 km)'
        WHEN depth_km <= 300 THEN 'Intermediate (70-300 km)'
        ELSE 'Deep (>300 km)'
    END AS depth_category,
    COUNT(*) AS event_count,
    ROUND(AVG(mag), 3) AS average_magnitude
FROM {TABLE_SQL}
GROUP BY
    CASE
        WHEN depth_km IS NULL THEN 'Missing'
        WHEN depth_km < 0 THEN 'Review - negative'
        WHEN depth_km < 70 THEN 'Shallow (<70 km)'
        WHEN depth_km <= 300 THEN 'Intermediate (70-300 km)'
        ELSE 'Deep (>300 km)'
    END
ORDER BY event_count DESC
""")

display(depth_categories)


,total_records,unique_event_ids,records_with_magnitude,magnitude_5_plus,magnitude_7_plus,tsunami_flagged_events,earliest_event_utc,latest_event_utc
0,739554,739554,739497,9110.0,71.0,585.0,2021-10-22 13:22:28,2026-10-22 13:13:42


,magnitude_band,event_count,percent_of_records
0,Missing,57,0.01
1,Below 2.0,525542,71.06
2,2.0-2.9,110351,14.92
3,3.0-3.9,23138,3.13
4,4.0-4.9,71356,9.65
5,5.0-5.9,8448,1.14
6,6.0-6.9,591,0.08
7,7.0+,71,0.01


,depth_category,event_count,average_magnitude
0,Shallow (<70 km),642745,1.571
1,Intermediate (70-300 km),65744,2.504
2,Review - negative,25047,1.009
3,Deep (>300 km),6018,4.360


In [55]:
# Data Quality Summary

data_quality_summary = run_query(f"""
SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN id IS NULL THEN 1 ELSE 0 END) AS missing_ids,
    SUM(CASE WHEN time IS NULL THEN 1 ELSE 0 END) AS missing_timestamps,
    SUM(
        CASE
            WHEN latitude IS NULL OR latitude NOT BETWEEN -90 AND 90
            THEN 1 ELSE 0
        END
    ) AS invalid_latitudes,
    SUM(
        CASE
            WHEN longitude IS NULL OR longitude NOT BETWEEN -180 AND 180
            THEN 1 ELSE 0
        END
    ) AS invalid_longitudes,
    SUM(CASE WHEN mag IS NULL THEN 1 ELSE 0 END) AS missing_magnitudes,
    SUM(CASE WHEN depth_km IS NULL THEN 1 ELSE 0 END) AS missing_depths,
    SUM(CASE WHEN depth_km < 0 THEN 1 ELSE 0 END) AS negative_depths
FROM {TABLE_SQL}
""")

display(data_quality_summary)

,total_records,missing_ids,missing_timestamps,invalid_latitudes,invalid_longitudes,missing_magnitudes,missing_depths,negative_depths
0,739554,0.0,0.0,0.0,0.0,57.0,0.0,25047.0


## 26. Export results to CSV

In [ ]:

expected_result_names = [
    "top_10_strongest",
    "top_10_deepest",
    "shallow_high_magnitude",
    "average_magnitude_by_type",
    "yearly_counts",
    "monthly_counts_by_name",
    "monthly_counts_by_period",
    "weekday_counts",
    "hourly_counts",
    "network_activity",
    "status_counts",
    "event_type_counts",
    "data_type_token_counts",
    "high_station_coverage_summary",
    "high_station_coverage_events",
    "tsunami_by_year",
    "alert_level_counts",
    "year_over_year_growth",
    "magnitude_by_tsunami_flag",
    "network_quality",
    "consecutive_nearby_events",
    "deep_focus_coordinate_bins",
    "overall_kpis",
    "magnitude_bands",
    "depth_categories",
    "data_quality_summary",
]
missing_result_names = [
    name for name in expected_result_names if name not in globals()
]
if missing_result_names:
    raise RuntimeError(
        "Cannot export SQL results because these variables are undefined: "
        + ", ".join(missing_result_names)
        + ". Run all analysis cells from the beginning in notebook order."
    )

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "processed").is_dir() and PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "sql_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

result_tables = {
    "top_10_strongest": top_10_strongest,
    "top_10_deepest": top_10_deepest,
    "shallow_high_magnitude": shallow_high_magnitude,
    "average_magnitude_by_type": average_magnitude_by_type,
    "yearly_counts": yearly_counts,
    "monthly_counts_by_name": monthly_counts_by_name,
    "monthly_counts_by_period": monthly_counts_by_period,
    "weekday_counts": weekday_counts,
    "hourly_counts": hourly_counts,
    "network_activity": network_activity,
    "status_counts": status_counts,
    "event_type_counts": event_type_counts,
    "data_type_token_counts": data_type_token_counts,
    "high_station_coverage_summary": high_station_coverage_summary,
    "high_station_coverage_events": high_station_coverage_events,
    "tsunami_by_year": tsunami_by_year,
    "alert_level_counts": alert_level_counts,
    "year_over_year_growth": year_over_year_growth,
    "magnitude_by_tsunami_flag": magnitude_by_tsunami_flag,
    "network_quality": network_quality,
    "consecutive_nearby_events": consecutive_nearby_events,
    "deep_focus_coordinate_bins": deep_focus_coordinate_bins,
    "overall_kpis": overall_kpis,
    "magnitude_bands": magnitude_bands,
    "depth_categories": depth_categories,
    "data_quality_summary": data_quality_summary,
}

saved_files = []
for result_name, result_df in result_tables.items():
    if isinstance(result_df, pd.DataFrame):
        file_path = OUTPUT_DIR / f"{result_name}_{run_timestamp}.csv"
        result_df.to_csv(file_path, index=False, encoding="utf-8")
        saved_files.append(str(file_path))

metadata = {
    "project": "Global Seismic Trends: Data-Driven Earthquake Insights",
    "notebook": "05_sql_analysis_updated.ipynb",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "database": DATABASE_NAME,
    "table": TABLE_NAME,
    "database_row_count": int(row_count),
    "column_count": int(len(column_names)),
    "alert_column_available": "alert" in column_names,
    "result_tables_exported": len(saved_files),
    "exported_files": saved_files,
    "excluded_tasks": [4, 11, 12, 13, 17, 21, 22, 24, 25, 26],
    "conditional_task": 20,
    "interpretation_note": "Results are descriptive. Country/continent, casualty, economic-loss, and official-region analyses require additional documented datasets."
}
metadata_path = OUTPUT_DIR / f"sql_analysis_metadata_{run_timestamp}.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False, default=str)

print(f"Exported {len(saved_files)} CSV result tables to: {OUTPUT_DIR}")
print("Metadata saved to:", metadata_path)
for file_path in saved_files:
    print("-", file_path)


Exported 26 CSV result tables to: D:\global seismic trends\data\processed\sql_analysis
Metadata saved to: D:\global seismic trends\data\processed\sql_analysis\sql_analysis_metadata_20261023T130010Z.json
- D:\global seismic trends\data\processed\sql_analysis\top_10_strongest_20261023T130010Z.csv
- D:\global seismic trends\data\processed\sql_analysis\top_10_deepest_20261023T130010Z.csv
- D:\global seismic trends\data\processed\sql_analysis\shallow_high_magnitude_20261023T130010Z.csv
- D:\global seismic trends\data\processed\sql_analysis\average_magnitude_by_type_20261023T130010Z.csv
- D:\global seismic trends\data\processed\sql_analysis\yearly_counts_20261023T130010Z.csv
- D:\global seismic trends\data\processed\sql_analysis\monthly_counts_by_name_20261023T130010Z.csv
- D:\global seismic trends\data\processed\sql_analysis\monthly_counts_by_period_20261023T130010Z.csv
- D:\global seismic trends\data\processed\sql_analysis\weekday_counts_20261023T130010Z.csv
- D:\global seismic trends\data